<center><h1>1-ab: Introduction to Neural Networks</h1></center>

<center><h2><a href="https://rdfia.github.io/">Course link</a></h2></center>

# Warning :
# Do "File -> Save a copy in Drive" before you start modifying the notebook, otherwise your modifications will not be saved.


In [ ]:
!wget https://github.com/rdfia/rdfia.github.io/raw/master/data/2-ab.zip
!unzip -j 2-ab.zip
!wget https://github.com/rdfia/rdfia.github.io/raw/master/code/2-ab/utils-data.py

In [ ]:
import math
import torch
from torch.autograd import Variable
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%run 'utils-data.py'

# Part 1 : Forward and backward passes "by hands"

In [ ]:
def init_params(nx, nh, ny):
    """
    nx, nh, ny: integers
    out params: dictionnary
    """
    params = {}

    #####################
    ## Your code here  ##
    #####################
    # fill values for Wh, Wy, bh, by

    Wh = torch.randn(nh,nx) * 0.3
    Wy = torch.randn(ny,nh) * 0.3
    bh = torch.zeros(nh)
    by = torch.zeros(ny)

    params["Wh"] = Wh
    params["Wy"] = Wy
    params["bh"] = bh
    params["by"] = by

    ####################
    ##      END        #
    ####################
    return params

In [ ]:
def forward(params, X):
    """
    params: dictionnary
    X: (n_batch, dimension)
    """
    bsize = X.size(0)
    nh = params['Wh'].size(0)
    ny = params['Wy'].size(0)
    outputs = {}

    #####################
    ## Your code here  ##
    #####################
    # fill values for X, htilde, h, ytilde, yhat

    htilde = torch.mm(X,params['Wh'].t()) + params['bh'].repeat(bsize,1)
    h = torch.tanh(htilde)
    ytilde = torch.mm(h,params['Wy'].t()) + params['by'].repeat(bsize,1)
    yhat = torch.softmax(ytilde, dim = 1)

    outputs["X"] = X
    outputs["htilde"] = htilde
    outputs["h"] = h
    outputs["ytilde"] = ytilde
    outputs["yhat"] = yhat

    ####################
    ##      END        #
    ####################

    return outputs['yhat'], outputs

In [ ]:
def loss_accuracy(Yhat, Y):

    #####################
    ## Your code here  ##
    #####################

    L = -torch.mean(torch.sum(Y * torch.log(Yhat)))
    _,indsY = torch.max(Y,1)
    _,indsYhat = torch.max(Yhat,1)
    acc = 100 * torch.sum(indsY == indsYhat) / indsY.size(0)

    ####################
    ##      END        #
    ####################

    return L, acc

In [ ]:
from weakref import WeakKeyDictionary
def backward(params, outputs, Y):
    bsize = Y.shape[0]
    grads = {}

    #####################
    ## Your code here  ##
    #####################
    # fill values for Wy, Wh, by, bh
    dytilde = (outputs['yhat'] - Y)
    dWy = torch.mm(dytilde.t(), outputs['h'])
    dby = torch.sum(dytilde, dim=0).t()

    dh = torch.mm(dytilde,params['Wy'])
    dhtilde = dh * (1 - outputs['h'] ** 2)
    dWh = torch.mm(dhtilde.t(), outputs['X'])
    dbh = torch.sum(dhtilde).t()

    grads["Wy"] = dWy/bsize
    grads["Wh"] = dWh/bsize
    grads["by"] = dby/bsize
    grads["bh"] = dbh/bsize

    ####################
    ##      END        #
    ####################
    return grads

In [ ]:
def sgd(params, grads, eta):

    #####################
    ## Your code here  ##
    #####################
    # update the params values

    with torch.no_grad():
      params["Wh"] -= eta * grads['Wh']
      params["Wy"] -= eta * grads['Wy']
      params["bh"] -= eta * grads['bh']
      params["by"] -= eta * grads['by']

    ####################
    ##      END        #
    ####################
    return params

## Global learning procedure "by hands"

Test learning rate

In [ ]:
# init
data = CirclesData()
data.plot_data()
N = data.Xtrain.shape[0]
Nbatch = 10
nx = data.Xtrain.shape[1]
nh = 10
ny = data.Ytrain.shape[1]
eta = [0.001,0.01,0.03,0.05,0.1]


curves = {lr: [[], [], [], []] for lr in eta}

for lr in eta :
  params = init_params(nx, nh, ny)

  # epoch
  for iteration in range(400):

      # permute
      perm = np.random.permutation(N)
      Xtrain = data.Xtrain[perm, :]
      Ytrain = data.Ytrain[perm, :]

      #####################
      ## Your code here  ##
      #####################
      # batches
      for j in range(N // Nbatch):

          indsBatch = range(j * Nbatch, (j+1) * Nbatch)
          X = Xtrain[indsBatch, :]
          Y = Ytrain[indsBatch, :]

          # write the optimization algorithm on the batch (X,Y)
          # using the functions: forward, loss_accuracy, backward, sgd
          yhat, outputs = forward(params,X)
          L, acc = loss_accuracy(yhat,Y)
          grads = backward(params, outputs, Y)
          sgd(params, grads, lr)

      ####################
      ##      END        #
      ####################


      Yhat_train, _ = forward(params, data.Xtrain)
      Yhat_test, _ = forward(params, data.Xtest)
      Ltrain, acctrain = loss_accuracy(Yhat_train, data.Ytrain)
      Ltest, acctest = loss_accuracy(Yhat_test, data.Ytest)
      Ygrid, _ = forward(params, data.Xgrid)

      title = 'Iter {}: Acc train {:.1f}% ({:.2f}), acc test {:.1f}% ({:.2f})'.format(iteration, acctrain, Ltrain, acctest, Ltest)
      #print(title)
      data.plot_data_with_grid(Ygrid.detach(), title)
      curves[lr][0].append(acctrain.detach())
      curves[lr][1].append(acctest.detach())
      curves[lr][2].append(Ltrain.detach())
      curves[lr][3].append(Ltest.detach())

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 10))
for lr, curve in curves.items():
    axes[0].plot(curve[0], label=f"LR={lr}")
    axes[1].plot(curve[1], label=f"LR={lr}")
    axes[2].plot(curve[2], label=f"LR={lr}")
    axes[3].plot(curve[3], label=f"LR={lr}")

axes[0].set_title("Accuracy (Train)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].set_title("Accuracy (test)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].set_title("Loss (Train)")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Loss")
axes[2].legend()

axes[3].set_title("Loss (test)")
axes[3].set_xlabel("Iteration")
axes[3].set_ylabel("Loss")
axes[3].legend()

plt.show()

Test Batch_size

In [ ]:
data = CirclesData()
data.plot_data()
N = data.Xtrain.shape[0]
batch_size = [10,20,50,100]
nx = data.Xtrain.shape[1]
nh = 10
ny = data.Ytrain.shape[1]
eta = 0.03

curves = {Nbatch: [[], [], [], []] for Nbatch in batch_size}
for Nbatch in batch_size :
  params = init_params(nx, nh, ny)

  # epoch
  for iteration in range(400):

      # permute
      perm = np.random.permutation(N)
      Xtrain = data.Xtrain[perm, :]
      Ytrain = data.Ytrain[perm, :]

      #####################
      ## Your code here  ##
      #####################
      # batches
      for j in range(N // Nbatch):

          indsBatch = range(j * Nbatch, (j+1) * Nbatch)
          X = Xtrain[indsBatch, :]
          Y = Ytrain[indsBatch, :]

          # write the optimization algorithm on the batch (X,Y)
          # using the functions: forward, loss_accuracy, backward, sgd
          yhat, outputs = forward(params,X)
          L, acc = loss_accuracy(yhat,Y)
          grads = backward(params, outputs, Y)
          sgd(params, grads, eta)

      ####################
      ##      END        #
      ####################


      Yhat_train, _ = forward(params, data.Xtrain)
      Yhat_test, _ = forward(params, data.Xtest)
      Ltrain, acctrain = loss_accuracy(Yhat_train, data.Ytrain)
      Ltest, acctest = loss_accuracy(Yhat_test, data.Ytest)
      Ygrid, _ = forward(params, data.Xgrid)

      title = 'Iter {}: Acc train {:.1f}% ({:.2f}), acc test {:.1f}% ({:.2f})'.format(iteration, acctrain, Ltrain, acctest, Ltest)
      #print(title)
      data.plot_data_with_grid(Ygrid.detach(), title)
      curves[Nbatch][0].append(acctrain.detach())
      curves[Nbatch][1].append(acctest.detach())
      curves[Nbatch][2].append(Ltrain.detach())
      curves[Nbatch][3].append(Ltest.detach())

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 10))
for Nbatch, curve in curves.items():
    axes[0].plot(curve[0], label=f"Batch_size={Nbatch}")
    axes[1].plot(curve[1], label=f"Batch_size={Nbatch}")
    axes[2].plot(curve[2], label=f"Batch_size={Nbatch}")
    axes[3].plot(curve[3], label=f"Batch_size={Nbatch}")

axes[0].set_title("Accuracy (Train)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].set_title("Accuracy (test)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].set_title("Loss (Train)")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Loss")
axes[2].legend()

axes[3].set_title("Loss (test)")
axes[3].set_xlabel("Iteration")
axes[3].set_ylabel("Loss")
axes[3].legend()

plt.show()

# Part 2 : Simplification of the backward pass with `torch.autograd`



In [ ]:
def init_params(nx, nh, ny):
    """
    nx, nh, ny: integers
    out params: dictionnary
    """
    params = {}

    #####################
    ## Your code here  ##
    #####################
    # fill values for Wh, Wy, bh, by
    # activaye autograd on the network weights

    Wh = torch.randn(nh,nx, requires_grad = True)
    Wy = torch.randn(ny,nh, requires_grad = True)
    bh = torch.zeros(nh, requires_grad = True)
    by = torch.zeros(ny, requires_grad = True)

    params["Wh"] = Wh
    params["Wy"] = Wy
    params["bh"] = bh
    params["by"] = by

    ####################
    ##      END        #
    ####################
    return params

The function `forward` remains unchanged from previous part.

The function `backward` is no longer used because of "autograd".

In [ ]:
def sgd(params, eta):

    #####################
    ## Your code here  ##
    #####################
    # update the network weights
    # warning: use torch.no_grad()
    # and reset to zero the gradient accumulators
    with torch.no_grad():
      params["Wh"] -= eta * params["Wh"].grad
      params["Wy"] -= eta * params["Wy"].grad
      params["bh"] -= eta * params["bh"].grad
      params["by"] -= eta * params["by"].grad

      params['Wh'].grad.zero_()
      params['Wy'].grad.zero_()
      params['by'].grad.zero_()
      params['bh'].grad.zero_()
    ####################
    ##      END        #
    ####################
    return params

## Global learning procedure with autograd

Influence of learning rate

In [ ]:
data = CirclesData()
data.plot_data()
N = data.Xtrain.shape[0]
Nbatch = 10
nx = data.Xtrain.shape[1]
nh = 10
ny = data.Ytrain.shape[1]
eta = [0.001, 0.01, 0.03, 0.05, 0.1]
curves = {lr: [[], [], [], []] for lr in eta}

for lr in eta:
  params = init_params(nx, nh, ny)
  # epoch
  for iteration in range(400):

      # permute
      perm = np.random.permutation(N)
      Xtrain = data.Xtrain[perm, :]
      Ytrain = data.Ytrain[perm, :]

      #####################
      ## Your code here  ##
      #####################
      # batches
      for j in range(N // Nbatch):

          indsBatch = range(j * Nbatch, (j+1) * Nbatch)
          X = Xtrain[indsBatch, :]
          Y = Ytrain[indsBatch, :]

          # write the optimization algorithm on the batch (X,Y)
          # using the functions: forward, loss_accuracy, sgd
          # and the backward function with autograd

          yhat, outputs = forward(params, X)
          L, acc = loss_accuracy(yhat, Y)
          L.backward()
          params = sgd(params, lr)
      ####################
      ##      END        #
      ####################


      Yhat_train, _ = forward(params, data.Xtrain)
      Yhat_test, _ = forward(params, data.Xtest)
      Ltrain, acctrain = loss_accuracy(Yhat_train, data.Ytrain)
      Ltest, acctest = loss_accuracy(Yhat_test, data.Ytest)
      Ygrid, _ = forward(params, data.Xgrid)

      title = 'Iter {}: Acc train {:.1f}% ({:.2f}), acc test {:.1f}% ({:.2f})'.format(iteration, acctrain, Ltrain, acctest, Ltest)
      #print(title)
      # detach() is used to remove the predictions from the computational graph in autograd
      #data.plot_data_with_grid(Ygrid.detach(), title)

      curves[lr][0].append(acctrain)
      curves[lr][1].append(acctest)
      curves[lr][2].append(Ltrain.detach().numpy())
      curves[lr][3].append(Ltest.detach().numpy())

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 10))
for lr, curve in curves.items():
    axes[0].plot(curve[0], label=f"LR={lr}")
    axes[1].plot(curve[1], label=f"LR={lr}")
    axes[2].plot(curve[2], label=f"LR={lr}")
    axes[3].plot(curve[3], label=f"LR={lr}")

axes[0].set_title("Accuracy (Train)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].set_title("Accuracy (test)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].set_title("Loss (Train)")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Loss")
axes[2].legend()

axes[3].set_title("Loss (test)")
axes[3].set_xlabel("Iteration")
axes[3].set_ylabel("Loss")
axes[3].legend()

plt.show()

Influence of batch_size

In [ ]:
# init
data = CirclesData()
data.plot_data()
N = data.Xtrain.shape[0]
batch_size = [10,20,50,100]
nx = data.Xtrain.shape[1]
nh = 10
ny = data.Ytrain.shape[1]
eta = 0.03
curves = {Nbatch: [[], [], [], []] for Nbatch in batch_size}

for Nbatch in batch_size:
  params = init_params(nx, nh, ny)
  # epoch
  for iteration in range(400):

      # permute
      perm = np.random.permutation(N)
      Xtrain = data.Xtrain[perm, :]
      Ytrain = data.Ytrain[perm, :]

      #####################
      ## Your code here  ##
      #####################
      # batches
      for j in range(N // Nbatch):

          indsBatch = range(j * Nbatch, (j+1) * Nbatch)
          X = Xtrain[indsBatch, :]
          Y = Ytrain[indsBatch, :]

          # write the optimization algorithm on the batch (X,Y)
          # using the functions: forward, loss_accuracy, sgd
          # and the backward function with autograd

          yhat, outputs = forward(params, X)
          L, acc = loss_accuracy(yhat, Y)
          L.backward()
          params = sgd(params, eta)
      ####################
      ##      END        #
      ####################


      Yhat_train, _ = forward(params, data.Xtrain)
      Yhat_test, _ = forward(params, data.Xtest)
      Ltrain, acctrain = loss_accuracy(Yhat_train, data.Ytrain)
      Ltest, acctest = loss_accuracy(Yhat_test, data.Ytest)
      Ygrid, _ = forward(params, data.Xgrid)

      title = 'Iter {}: Acc train {:.1f}% ({:.2f}), acc test {:.1f}% ({:.2f})'.format(iteration, acctrain, Ltrain, acctest, Ltest)
      #print(title)
      # detach() is used to remove the predictions from the computational graph in autograd
      #data.plot_data_with_grid(Ygrid.detach(), title)

      curves[Nbatch][0].append(acctrain)
      curves[Nbatch][1].append(acctest)
      curves[Nbatch][2].append(Ltrain.detach().numpy())
      curves[Nbatch][3].append(Ltest.detach().numpy())

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 10))
for Nbatch, curve in curves.items():
    axes[0].plot(curve[0], label=f"Batch_size={Nbatch}")
    axes[1].plot(curve[1], label=f"Batch_size={Nbatch}")
    axes[2].plot(curve[2], label=f"Batch_size={Nbatch}")
    axes[3].plot(curve[3], label=f"Batch_size={Nbatch}")

axes[0].set_title("Accuracy (Train)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].set_title("Accuracy (test)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].set_title("Loss (Train)")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Loss")
axes[2].legend()

axes[3].set_title("Loss (test)")
axes[3].set_xlabel("Iteration")
axes[3].set_ylabel("Loss")
axes[3].legend()

plt.show()

# Part 3 : Simplification of the forward pass with `torch.nn`

`init_params` and `forward` are replaced by the `init_model` function which defines the network architecture and the loss.

In [ ]:
def init_model(nx, nh, ny):

    #####################
    ## Your code here  ##
    #####################

    model = torch.nn.Sequential(
        torch.nn.Linear(nx, nh),
        torch.nn.Tanh(),
        torch.nn.Linear(nh, ny),
    )
    loss = torch.nn.CrossEntropyLoss()

    ####################
    ##      END        #
    ####################

    return model, loss

In [ ]:
def loss_accuracy(loss, Yhat, Y):

    #####################
    ## Your code here  ##
    #####################
    # call the loss function
    L = loss(Yhat,Y)
    acc = 100 * torch.mean((Yhat.argmax(1) == Y.argmax(1)).float())

    ####################
    ##      END        #
    ####################

    return L, acc

In [ ]:
def sgd(model, eta):

    #####################
    ## Your code here  ##
    #####################
    # update the network weights
    # warning: use torch.no_grad()
    # and reset to zero the gradient accumulators
    with torch.no_grad():
      for param in model.parameters():
        param -= eta * param.grad
      model.zero_grad()
    ####################
    ##      END        #
    ####################
    return model

## Global learning procedure with autograd and `torch.nn`

Influence of the learning rate

In [ ]:
# init
data = CirclesData()
data.plot_data()
N = data.Xtrain.shape[0]
Nbatch = 10
nx = data.Xtrain.shape[1]
nh = 10
ny = data.Ytrain.shape[1]
eta = [0.001, 0.01, 0.03, 0.05, 0.1]

curves = {lr: [[], [], [], []] for lr in eta}

for lr in eta:
  model, loss = init_model(nx, nh, ny)
# epoch
  for iteration in range(400):

      # permute
      perm = np.random.permutation(N)
      Xtrain = data.Xtrain[perm, :]
      Ytrain = data.Ytrain[perm, :]

      #####################
      ## Your code here  ##
      #####################
      # batches
      for j in range(N // Nbatch):

          indsBatch = range(j * Nbatch, (j+1) * Nbatch)
          X = Xtrain[indsBatch, :]
          Y = Ytrain[indsBatch, :]

          # write the optimization algorithm on the batch (X,Y)
          # using the functions: loss_accuracy, sgd
          # the forward with the predict method from the model
          # and the backward function with autograd
          Yhat = model(X)
          L, acc = loss_accuracy(loss, Yhat, Y)
          L.backward()
          model = sgd(model, lr)

      ####################
      ##      END        #
      ####################


      Yhat_train = model(data.Xtrain)
      Yhat_test = model(data.Xtest)
      Ltrain, acctrain = loss_accuracy(loss, Yhat_train, data.Ytrain)
      Ltest, acctest = loss_accuracy(loss, Yhat_test, data.Ytest)
      Ygrid = model(data.Xgrid)
      torch.nn.Softmax()(Ygrid)

      title = 'Iter {}: Acc train {:.1f}% ({:.2f}), acc test {:.1f}% ({:.2f})'.format(iteration, acctrain, Ltrain, acctest, Ltest)
      #print(title)
      #data.plot_data_with_grid(torch.nn.Softmax(dim=1)(Ygrid.detach()), title)

      curves[lr][0].append(acctrain)
      curves[lr][1].append(acctest)
      curves[lr][2].append(Ltrain.detach().numpy())
      curves[lr][3].append(Ltest.detach().numpy())

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 10))
for lr, curve in curves.items():
    axes[0].plot(curve[0], label=f"LR={lr}")
    axes[1].plot(curve[1], label=f"LR={lr}")
    axes[2].plot(curve[2], label=f"LR={lr}")
    axes[3].plot(curve[3], label=f"LR={lr}")

axes[0].set_title("Accuracy (Train)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].set_title("Accuracy (test)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].set_title("Loss (Train)")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Loss")
axes[2].legend()

axes[3].set_title("Loss (test)")
axes[3].set_xlabel("Iteration")
axes[3].set_ylabel("Loss")
axes[3].legend()

plt.show()

Influence of batch_size

In [ ]:
 #init
data = CirclesData()
data.plot_data()
N = data.Xtrain.shape[0]
batch_size = [10,20,50,100]
nx = data.Xtrain.shape[1]
nh = 10
ny = data.Ytrain.shape[1]
eta = 0.03

curves = {Nbatch: [[], [], [], []] for Nbatch in batch_size}

for Nbatch in batch_size:
  model, loss = init_model(nx, nh, ny)
# epoch
  for iteration in range(400):

      # permute
      perm = np.random.permutation(N)
      Xtrain = data.Xtrain[perm, :]
      Ytrain = data.Ytrain[perm, :]

      #####################
      ## Your code here  ##
      #####################
      # batches
      for j in range(N // Nbatch):

          indsBatch = range(j * Nbatch, (j+1) * Nbatch)
          X = Xtrain[indsBatch, :]
          Y = Ytrain[indsBatch, :]

          # write the optimization algorithm on the batch (X,Y)
          # using the functions: loss_accuracy, sgd
          # the forward with the predict method from the model
          # and the backward function with autograd
          Yhat = model(X)
          L, acc = loss_accuracy(loss, Yhat, Y)
          L.backward()
          model = sgd(model, eta)

      ####################
      ##      END        #
      ####################


      Yhat_train = model(data.Xtrain)
      Yhat_test = model(data.Xtest)
      Ltrain, acctrain = loss_accuracy(loss, Yhat_train, data.Ytrain)
      Ltest, acctest = loss_accuracy(loss, Yhat_test, data.Ytest)
      Ygrid = model(data.Xgrid)
      torch.nn.Softmax()(Ygrid)

      title = 'Iter {}: Acc train {:.1f}% ({:.2f}), acc test {:.1f}% ({:.2f})'.format(iteration, acctrain, Ltrain, acctest, Ltest)
      #print(title)
      #data.plot_data_with_grid(torch.nn.Softmax(dim=1)(Ygrid.detach()), title)

      curves[Nbatch][0].append(acctrain)
      curves[Nbatch][1].append(acctest)
      curves[Nbatch][2].append(Ltrain.detach().numpy())
      curves[Nbatch][3].append(Ltest.detach().numpy())

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 10))
for Nbatch, curve in curves.items():
    axes[0].plot(curve[0], label=f"batch_size={Nbatch}")
    axes[1].plot(curve[1], label=f"batch_size={Nbatch}")
    axes[2].plot(curve[2], label=f"batch_size={Nbatch}")
    axes[3].plot(curve[3], label=f"batch_size={Nbatch}")

axes[0].set_title("Accuracy (Train)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].set_title("Accuracy (test)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].set_title("Loss (Train)")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Loss")
axes[2].legend()

axes[3].set_title("Loss (test)")
axes[3].set_xlabel("Iteration")
axes[3].set_ylabel("Loss")
axes[3].legend()

plt.show()

# Part 4 : Simplification of the SGD with `torch.optim`

In [ ]:
def init_model(nx, nh, ny, eta):

    #####################
    ## Your code here  ##
    #####################

    model = torch.nn.Sequential(
        torch.nn.Linear(nx, nh),
        torch.nn.Tanh(),
        torch.nn.Linear(nh, ny),
    )
    loss = torch.nn.CrossEntropyLoss()
    optim = torch.optim.SGD(params = model.parameters(), lr = eta)

    ####################
    ##      END        #
    ####################

    return model, loss, optim

The `sgd` function is replaced by calling the `optim.zero_grad()` before the backward and `optim.step()` after.

## Algorithme global d'apprentissage (avec autograd, les couches `torch.nn` et `torch.optim`)

Influence of learning rate

In [ ]:
# init
data = CirclesData()
data.plot_data()
N = data.Xtrain.shape[0]
batch_size = [10,20,50,100]
nx = data.Xtrain.shape[1]
nh = 10
ny = data.Ytrain.shape[1]
eta = [0.001, 0.01, 0.03, 0.05, 0.1]
curves = {lr: [[], [], [], []] for lr in eta}
for lr in eta:
  model, loss, optim = init_model(nx, nh, ny, lr)

  # epoch
  for iteration in range(400):

      # permute
      perm = np.random.permutation(N)
      Xtrain = data.Xtrain[perm, :]
      Ytrain = data.Ytrain[perm, :]

      #####################
      ## Your code  here ##
      #####################
      # batches
      for j in range(N // Nbatch):

          indsBatch = range(j * Nbatch, (j+1) * Nbatch)
          X = Xtrain[indsBatch, :]
          Y = Ytrain[indsBatch, :]

          # write the optimization algorithm on the batch (X,Y)
          # using the functions: loss_accuracy
          # the forward with the predict method from the model
          # the backward function with autograd
          # and then an optimization step

          Yhat = model(X)
          L, acc = loss_accuracy(loss, Yhat, Y)
          L.backward()
          optim.step()
          optim.zero_grad()

      ####################
      ##      FIN        #
      ####################


      Yhat_train = model(data.Xtrain)
      Yhat_test = model(data.Xtest)
      Ltrain, acctrain = loss_accuracy(loss, Yhat_train, data.Ytrain)
      Ltest, acctest = loss_accuracy(loss, Yhat_test, data.Ytest)
      Ygrid = model(data.Xgrid)
      torch.nn.Softmax()(Ygrid)

      title = 'Iter {}: Acc train {:.1f}% ({:.2f}), acc test {:.1f}% ({:.2f})'.format(iteration, acctrain, Ltrain, acctest, Ltest)
      #print(title)
      #data.plot_data_with_grid(torch.nn.Softmax(dim=1)(Ygrid.detach()), title)

      curves[lr][0].append(acctrain)
      curves[lr][1].append(acctest)
      curves[lr][2].append(Ltrain.detach().numpy())
      curves[lr][3].append(Ltest.detach().numpy())

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 10))
for lr, curve in curves.items():
    axes[0].plot(curve[0], label=f"LR={lr}")
    axes[1].plot(curve[1], label=f"LR={lr}")
    axes[2].plot(curve[2], label=f"LR={lr}")
    axes[3].plot(curve[3], label=f"LR={lr}")

axes[0].set_title("Accuracy (Train)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].set_title("Accuracy (test)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].set_title("Loss (Train)")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Loss")
axes[2].legend()

axes[3].set_title("Loss (test)")
axes[3].set_xlabel("Iteration")
axes[3].set_ylabel("Loss")
axes[3].legend()

plt.show()

Influence of batch size

In [ ]:
# init
data = CirclesData()
data.plot_data()
N = data.Xtrain.shape[0]
batch_size = [10,20,50,100]
nx = data.Xtrain.shape[1]
nh = 10
ny = data.Ytrain.shape[1]
eta = 0.03
curves = {Nbatch: [[], [], [], []] for Nbatch in batch_size}
for Nbatch in batch_size:
  model, loss, optim = init_model(nx, nh, ny, eta)

  # epoch
  for iteration in range(400):

      # permute
      perm = np.random.permutation(N)
      Xtrain = data.Xtrain[perm, :]
      Ytrain = data.Ytrain[perm, :]

      #####################
      ## Your code  here ##
      #####################
      # batches
      for j in range(N // Nbatch):

          indsBatch = range(j * Nbatch, (j+1) * Nbatch)
          X = Xtrain[indsBatch, :]
          Y = Ytrain[indsBatch, :]

          # write the optimization algorithm on the batch (X,Y)
          # using the functions: loss_accuracy
          # the forward with the predict method from the model
          # the backward function with autograd
          # and then an optimization step

          Yhat = model(X)
          L, acc = loss_accuracy(loss, Yhat, Y)
          L.backward()
          optim.step()
          optim.zero_grad()

      ####################
      ##      FIN        #
      ####################


      Yhat_train = model(data.Xtrain)
      Yhat_test = model(data.Xtest)
      Ltrain, acctrain = loss_accuracy(loss, Yhat_train, data.Ytrain)
      Ltest, acctest = loss_accuracy(loss, Yhat_test, data.Ytest)
      Ygrid = model(data.Xgrid)
      torch.nn.Softmax()(Ygrid)

      title = 'Iter {}: Acc train {:.1f}% ({:.2f}), acc test {:.1f}% ({:.2f})'.format(iteration, acctrain, Ltrain, acctest, Ltest)
      #print(title)
      #data.plot_data_with_grid(torch.nn.Softmax(dim=1)(Ygrid.detach()), title)

      curves[Nbatch][0].append(acctrain)
      curves[Nbatch][1].append(acctest)
      curves[Nbatch][2].append(Ltrain.detach().numpy())
      curves[Nbatch][3].append(Ltest.detach().numpy())

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 10))
for Nbatch, curve in curves.items():
    axes[0].plot(curve[0], label=f"batch_size={Nbatch}")
    axes[1].plot(curve[1], label=f"batch_size={Nbatch}")
    axes[2].plot(curve[2], label=f"batch_size={Nbatch}")
    axes[3].plot(curve[3], label=f"batch_size={Nbatch}")

axes[0].set_title("Accuracy (Train)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].set_title("Accuracy (test)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].set_title("Loss (Train)")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Loss")
axes[2].legend()

axes[3].set_title("Loss (test)")
axes[3].set_xlabel("Iteration")
axes[3].set_ylabel("Loss")
axes[3].legend()

plt.show()

# Part 5 : MNIST

Apply the code from previous part code to the MNIST dataset.

In [ ]:
# init
data = MNISTData()
N = data.Xtrain.shape[0]
Nbatch = 100
nx = data.Xtrain.shape[1]
nh = 100
ny = data.Ytrain.shape[1]
eta = 0.03
model, loss, optim = init_model(nx, nh, ny, eta)

curves = [[], [], [], []]

In [ ]:
  # epoch
for iteration in range(400):
   # permute
    perm = np.random.permutation(N)
    Xtrain = data.Xtrain[perm, :]
    Ytrain = data.Ytrain[perm, :]

        #####################
        ## Your code here  ##
        #####################
        # batches
    for j in range(N // Nbatch):

        indsBatch = range(j * Nbatch, (j+1) * Nbatch)
        X = Xtrain[indsBatch, :]
        Y = Ytrain[indsBatch, :]

        Yhat = model(X)
        L, acc = loss_accuracy(loss, Yhat, Y)
        L.backward()
        optim.step()
        optim.zero_grad()
        ##      END        #
        ####################


    Yhat_train = model(data.Xtrain)
    Yhat_test = model(data.Xtest)
    Ltrain, acctrain = loss_accuracy(loss, Yhat_train, data.Ytrain)
    Ltest, acctest = loss_accuracy(loss, Yhat_test, data.Ytest)

    curves[0].append(acctrain)
    curves[1].append(acctest)
    curves[2].append(Ltrain.detach().numpy())
    curves[3].append(Ltest.detach().numpy())

fig = plt.figure()
plt.plot(curves[0], label="acc. train")
plt.plot(curves[1], label="acc. test")
plt.legend()
plt.show()
plt.plot(curves[2], label="loss train")
plt.plot(curves[3], label="loss test")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(data.Ytest.argmax(dim=1), Yhat_test.detach().argmax(dim=1))

print(cm, "CONFUSION MATRIX")

# Part 6: Bonus: SVM


Train a SVM model on the Circles dataset.

Ideas :
- First try a linear SVM (sklearn.svm.LinearSVC dans scikit-learn). Does it work well ? Why ?
- Then try more complex kernels (sklearn.svm.SVC). Which one is the best ? why ?
- Does the parameter C of regularization have an impact? Why ?

In [ ]:
# data
data = CirclesData()
Xtrain = data.Xtrain.numpy()
Ytrain = data.Ytrain[:, 0].numpy()

Xgrid = data.Xgrid.numpy()

Xtest = data.Xtest.numpy()
Ytest = data.Ytest[:, 0].numpy()

def plot_svm_predictions(data, predictions):
      plt.figure(2)
      plt.clf()
      plt.imshow(np.reshape(predictions, (40,40)))
      plt.plot(data._Xtrain[data._Ytrain[:,0] == 1,0]*10+20, data._Xtrain[data._Ytrain[:,0] == 1,1]*10+20, 'bo', label="Train")
      plt.plot(data._Xtrain[data._Ytrain[:,1] == 1,0]*10+20, data._Xtrain[data._Ytrain[:,1] == 1,1]*10+20, 'ro')
      plt.plot(data._Xtest[data._Ytest[:,0] == 1,0]*10+20, data._Xtest[data._Ytest[:,0] == 1,1]*10+20, 'b+', label="Test")
      plt.plot(data._Xtest[data._Ytest[:,1] == 1,0]*10+20, data._Xtest[data._Ytest[:,1] == 1,1]*10+20, 'r+')
      plt.xlim(0,39)
      plt.ylim(0,39)
      plt.clim(0.3,0.7)
      plt.draw()
      plt.pause(1e-3)

In [ ]:
import sklearn.svm

############################
### Your code here   #######
### Train the SVM    #######
## See https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
## and https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
############################

svm = None

###########################

In [ ]:
## Print results

Ytest_pred = svm.predict(Xtest)
accuracy = np.sum(Ytest == Ytest_pred) / len(Ytest)
print(f"Accuracy : {100 * accuracy:.2f}")
Ygrid_pred = svm.predict(Xgrid)
plot_svm_predictions(data, Ygrid_pred)